# MaterialForge: GRPO Training for Crystal Design

**Meta PyTorch OpenEnv Hackathon x Scaler School of Technology — Grand Finale**

This notebook trains an LLM to autonomously design crystalline structures using reinforcement learning (GRPO) with the MaterialForge environment.

| Component | Details |
|-----------|----------|
| **Environment** | MaterialForge — 8x8 atomic lattice, 4 atom species, 4 physical properties |
| **Algorithm** | GRPO (Group Relative Policy Optimization) via HuggingFace TRL |
| **Efficiency** | Unsloth for 4-bit QLoRA training |
| **Framework** | OpenEnv-compliant environment |
| **HF Space** | [ArshPathan/material_forge_env](https://huggingface.co/spaces/ArshPathan/material_forge_env) |

## 1. Installation

In [ ]:
%%capture
!pip install -Uq unsloth
!pip install -Uq trl>=0.18.0 transformers datasets accelerate peft
!pip install -Uq "openenv-core[core]>=0.2.3" aiofiles openai python-dotenv
!pip install -Uq matplotlib jmespath

## 2. Clone Environment & Setup Paths

We clone the MaterialForge repository to access the environment code directly.
The same environment is hosted live at: https://huggingface.co/spaces/ArshPathan/material_forge_env

In [ ]:
import os

REPO_URL = "https://huggingface.co/spaces/ArshPathan/material_forge_env"
REPO_DIR = "MaterialForge"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR}/ already exists, skipping clone.")

import sys
sys.path.insert(0, REPO_DIR)

print("Environment source ready.")

## 3. Verify Environment Works

Quick sanity check: reset the environment, take one action, confirm reward is computed.

In [ ]:
from environment.rubrics import HeuristicRewardRubric
from server.material_forge_env_environment import MaterialForgeEnvironment
from models import MaterialForgeAction

env = MaterialForgeEnvironment(rubric=HeuristicRewardRubric())
obs = env.reset(seed=42, difficulty="easy")
print(f"Target properties: {obs.target}")
print(f"Cost budget: {obs.cost_budget}")
print(f"Max steps: {obs.max_steps}")

action = MaterialForgeAction(action_type="place", row=3, col=3, atom="A")
obs = env.step(action)
print(f"\nAfter placing Metal at (3,3):")
print(f"  Reward: {obs.reward}")
print(f"  Phase: {obs.phase}")
print(f"  Properties: {obs.current_properties}")
print("\nEnvironment OK!")

## 4. TRL Environment Wrapper

TRL's `GRPOTrainer` with `environment_factory` requires a wrapper class that exposes **named tool methods** (not a generic `step()`). The trainer discovers these methods and presents them to the LLM as callable tools.

Our wrapper exposes three tools:
- `place_atom(row, col, atom)` — place a new atom on an empty cell
- `remove_atom(row, col)` — remove an existing atom
- `replace_atom(row, col, atom)` — swap an atom for a different species

Each tool internally calls the OpenEnv environment's standard `step()` method.

In [ ]:
import random
from typing import Optional

from environment.config import GRID_SIZE, PROPERTY_NAMES, ATOM_TYPES
from environment.rubrics import HeuristicRewardRubric
from models import MaterialForgeAction, MaterialForgeObservation
from server.material_forge_env_environment import MaterialForgeEnvironment


DIFFICULTIES = ["easy", "medium", "hard"]


class MaterialForgeTRLEnv:
    """TRL-compatible wrapper for MaterialForge crystal design."""

    def __init__(self):
        self.env = MaterialForgeEnvironment(rubric=HeuristicRewardRubric())
        self.reward = 0.0
        self.done = False
        self._obs = None
        self._best_reward = 0.0
        self._reward_history = []
        self._invalid_actions = 0
        self._total_actions = 0

    def reset(self, **kwargs) -> str:
        seed = random.randint(0, 99999)
        difficulty = random.choice(DIFFICULTIES)
        obs = self.env.reset(seed=seed, difficulty=difficulty)
        self._obs = obs
        self.reward = 0.0
        self.done = False
        self._best_reward = 0.0
        self._reward_history = []
        self._invalid_actions = 0
        self._total_actions = 0
        return self._format_observation(obs)

    def place_atom(self, row: int, col: int, atom: str) -> str:
        """Place an atom on an empty cell of the crystal lattice.

        Args:
            row: Row index on the 8x8 grid (0-7)
            col: Column index on the 8x8 grid (0-7)
            atom: Atom species - 'A' (Metal, hardness, cost 8), 'B' (Conductor, conductivity, cost 6), 'C' (Ceramic, thermal resistance, cost 4), or 'P' (Polymer, elasticity, cost 2)

        Returns:
            Updated lattice state with current properties, target gaps, and phase.
        """
        return self._do_step("place", row, col, atom)

    def remove_atom(self, row: int, col: int) -> str:
        """Remove an atom from the crystal lattice to free up budget.

        Args:
            row: Row index on the 8x8 grid (0-7)
            col: Column index on the 8x8 grid (0-7)

        Returns:
            Updated lattice state with current properties, target gaps, and phase.
        """
        return self._do_step("remove", row, col, None)

    def replace_atom(self, row: int, col: int, atom: str) -> str:
        """Replace an existing atom with a different species to adjust material properties.

        Args:
            row: Row index on the 8x8 grid (0-7)
            col: Column index on the 8x8 grid (0-7)
            atom: New atom species - 'A' (Metal), 'B' (Conductor), 'C' (Ceramic), or 'P' (Polymer)

        Returns:
            Updated lattice state with current properties, target gaps, and phase.
        """
        return self._do_step("replace", row, col, atom)

    def _do_step(self, action_type: str, row: int, col: int, atom: Optional[str]) -> str:
        if self.done:
            raise ValueError("Episode is over. The crystal design session has ended.")

        self._total_actions += 1
        row = max(0, min(int(row), GRID_SIZE - 1))
        col = max(0, min(int(col), GRID_SIZE - 1))

        if atom is not None:
            atom = str(atom).upper().strip()
            if atom not in ATOM_TYPES:
                atom = "A"

        # Pre-check: will this action actually change the grid?
        grid = self._obs.grid if self._obs else None
        if grid:
            cell = grid[row][col]
            if action_type == "place" and cell != ".":
                self._invalid_actions += 1
                return (
                    f"INVALID ACTION: Cannot place at ({row},{col}) — cell is already "
                    f"occupied by '{cell}'. Use replace_atom to change it, or pick an empty cell (marked '.').\n"
                    + self._format_observation(self._obs)
                )
            if action_type == "remove" and cell == ".":
                self._invalid_actions += 1
                return (
                    f"INVALID ACTION: Cannot remove from ({row},{col}) — cell is empty. "
                    f"Pick a cell that contains an atom.\n"
                    + self._format_observation(self._obs)
                )
            if action_type == "replace" and cell == ".":
                self._invalid_actions += 1
                return (
                    f"INVALID ACTION: Cannot replace at ({row},{col}) — cell is empty. "
                    f"Use place_atom instead.\n"
                    + self._format_observation(self._obs)
                )
            if action_type == "replace" and cell == atom:
                self._invalid_actions += 1
                return (
                    f"INVALID ACTION: Cell ({row},{col}) already contains '{atom}'. "
                    f"Pick a different atom species to replace it with.\n"
                    + self._format_observation(self._obs)
                )

        action = MaterialForgeAction(action_type=action_type, row=row, col=col, atom=atom)
        obs = self.env.step(action)
        self._obs = obs
        step_reward = obs.reward if obs.reward is not None else 0.0
        self._reward_history.append(step_reward)
        self._best_reward = max(self._best_reward, step_reward)
        self.reward = self._best_reward
        self.done = obs.done
        return self._format_observation(obs)

    def _format_observation(self, obs) -> str:
        grid_lines = []
        for r, row in enumerate(obs.grid):
            grid_lines.append(f"  {r}: {' '.join(cell if cell != '.' else '.' for cell in row)}")
        grid_str = "\n".join(grid_lines)

        gaps = {}
        for prop in PROPERTY_NAMES:
            t = obs.target.get(prop, 0.0)
            c = obs.current_properties.get(prop, 0.0)
            gaps[prop] = round(t - c, 1)

        props_str = ", ".join(
            f"{p}={obs.current_properties.get(p, 0):.1f}/{obs.target.get(p, 0):.1f}(gap:{gaps[p]:+.1f})"
            for p in PROPERTY_NAMES
        )

        sb = obs.score_breakdown or {}
        stability = sb.get("structural_stability", 0.0)
        order = sb.get("lattice_order_index", 0.0)

        return (
            f"Step {obs.step_number}/{obs.max_steps} | "
            f"Cost: {obs.total_cost:.0f}/{obs.cost_budget:.0f} | "
            f"Phase: {obs.phase} | "
            f"Reward: {obs.reward:.4f}\n"
            f"Properties: {props_str}\n"
            f"Stability: {stability:.3f} | Order: {order:.3f}\n"
            f"Grid (row: cols 0-7):\n{grid_str}"
        )


print("MaterialForgeTRLEnv defined.")

### Test the wrapper

In [ ]:
test_env = MaterialForgeTRLEnv()
init_obs = test_env.reset()
print("=== Initial State ===")
print(init_obs)

print("\n=== After place_atom(3, 3, 'A') ===")
result = test_env.place_atom(3, 3, "A")
print(result)
print(f"\nReward stored: {test_env.reward:.4f}")
print(f"Done: {test_env.done}")

print("\n=== After place_atom(3, 4, 'B') ===")
result = test_env.place_atom(3, 4, "B")
print(result)
print(f"Reward: {test_env.reward:.4f}")

## 5. Reward Function

In [ ]:
def reward_func(environments, **kwargs) -> list[float]:
    """Reward = best_reward + spatial_bonus + phase_bonus - invalid_penalty."""
    rewards = []
    for env in environments:
        best = env._best_reward
        total = max(env._total_actions, 1)

        # Penalty for invalid actions
        invalid_ratio = env._invalid_actions / total
        invalid_penalty = 0.3 * invalid_ratio

        # Spatial diversity bonus: reward using multiple rows and columns
        spatial_bonus = 0.0
        if env._obs is not None:
            grid = env._obs.grid
            rows_used = set()
            cols_used = set()
            for r, row in enumerate(grid):
                for c, cell in enumerate(row):
                    if cell != ".":
                        rows_used.add(r)
                        cols_used.add(c)
            # Bonus for spreading across rows AND columns (max 0.10)
            row_spread = min(len(rows_used) / 4.0, 1.0)  # 4+ rows = full bonus
            col_spread = min(len(cols_used) / 4.0, 1.0)  # 4+ cols = full bonus
            spatial_bonus = 0.10 * (row_spread * col_spread)

        # Phase bonus
        phase_bonus = 0.0
        if env._obs is not None:
            phase = env._obs.phase
            if phase == "crystalline":
                phase_bonus = 0.15
            elif phase == "polycrystalline":
                phase_bonus = 0.05

        final = best + spatial_bonus + phase_bonus - invalid_penalty
        rewards.append(max(min(final, 1.0), 0.0))
    return rewards

print("Reward function: best_reward + spatial_bonus + phase_bonus - invalid_penalty")

## 6. Training Dataset

We create a prompt dataset where each entry instructs the LLM to design a crystal.
The environment generates random targets on each `reset()`, so every episode is unique.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are an expert materials scientist designing crystal structures on an 8x8 atomic lattice.

Your goal: place atoms to match target physical properties while staying within budget.

Available atom species:
- A (Metal): Primary contribution to hardness, cost 8 per atom
- B (Conductor): Primary contribution to conductivity, cost 6 per atom
- C (Ceramic): Primary contribution to thermal resistance, cost 4 per atom
- P (Polymer): Primary contribution to elasticity, cost 2 per atom

Strategy:
- START from the center of the grid (around row 3-4, col 3-4) and expand outward in all directions
- Build 2D clusters, NOT single rows or columns — atoms need neighbors in multiple directions for bonding
- Group same-type atoms together in 2x2 or 3x3 blocks for maximum bonding bonuses
- After each placement, READ the property gaps and choose the atom type that closes the LARGEST gap
- Aim for crystalline phase: fill a compact rectangular region, not scattered atoms
- Stay within the cost budget — use cheaper atoms (P=2, C=4) when budget is tight
- Use replace_atom to fix overshooting properties, remove_atom if over budget

Use the tools place_atom, remove_atom, and replace_atom to build your crystal."""

USER_PROMPT = "Design a crystal lattice that matches the target material properties. Start placing atoms from the center of the grid (row 3-4, col 3-4) and expand outward in all directions. Analyze the property gaps after each action and choose your next atom strategically."

NUM_EPISODES = 200

dataset = Dataset.from_dict({
    "prompt": [
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT},
        ]
    ] * NUM_EPISODES
})

print(f"Dataset created: {len(dataset)} episodes")
print(f"Sample prompt roles: {[m['role'] for m in dataset[0]['prompt']]}")

## 7. Load Model with Unsloth

We use Unsloth for memory-efficient 4-bit QLoRA fine-tuning. Starting with `Qwen/Qwen3-0.6B` which fits on a free Colab T4 GPU. For better results with more compute, switch to `Qwen/Qwen3-1.7B` or `Qwen/Qwen2.5-3B-Instruct`.

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "Qwen/Qwen3-0.6B"  # Change to Qwen3-1.7B or Qwen2.5-3B-Instruct for better results
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

# --- Fix Unsloth + TRL compatibility for Qwen3 tool calling ---
# Unsloth modifies the chat template, breaking TRL's string-comparison checks.
# We load TRL's own Qwen3 training template (prefix-preserving, has {% generation %}
# markers) and response schema directly.
from pathlib import Path
import trl.chat_template_utils as _ctu

# 1) Set the response schema so TRL can parse <tool_call> blocks
tokenizer.response_schema = {
    "x-regex": r"^(?:<think>\n?(?:(?P<reasoning_content>.*?\S.*?)\n?|[\s]*)</think>\s*)?(?P<content>.*?)(?:\n(?=<tool_call>))?(?=(?:<tool_call>|<\|im_end\|>|$))(?P<tool_calls>(?:<tool_call>.+?</tool_call>\s*)+)?\s*(?:<\|im_end\|>|$)",
    "type": "object",
    "properties": {
        "role": {"const": "assistant"},
        "content": {"type": "string"},
        "reasoning_content": {"type": "string"},
        "tool_calls": {
            "type": "array",
            "x-regex-iterator": r"<tool_call>\s*(.+?)\s*</tool_call>",
            "items": {
                "x-parser": "json",
                "x-parser-args": {"transform": "{type: 'function', function: @}"},
                "type": "object",
                "properties": {
                    "type": {"const": "function"},
                    "function": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "arguments": {"type": "object", "additionalProperties": {}},
                        },
                    },
                },
            },
        },
    },
}

# 2) Replace Unsloth's modified template with TRL's training-safe Qwen3 template
_templates_dir = Path(_ctu.__file__).parent / "chat_templates"
tokenizer.chat_template = (_templates_dir / "qwen3_training.jinja").read_text()

print(f"Model loaded: {MODEL_NAME}")
print(f"Trainable params: {model.print_trainable_parameters()}")
print(f"Chat template: TRL qwen3_training.jinja (prefix-preserving)")
print(f"Response schema: Qwen3 tool-call parser")

## 8. GRPO Training

The `GRPOTrainer` with `environment_factory` handles the full multi-turn loop:
1. Generates completions from the LLM
2. Parses tool calls (`place_atom`, `remove_atom`, `replace_atom`)
3. Executes them against the MaterialForge environment
4. Feeds observations back to the model
5. Collects rewards and updates the policy via GRPO

In [ ]:
from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = "materialforge-grpo-checkpoints"

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,

    # Generation
    num_generations=4,
    max_completion_length=2048,
    temperature=1.0,

    # Optimization
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,

    # Logging
    logging_steps=1,
    log_completions=True,
    num_completions_to_print=1,

    # Tool calling
    chat_template_kwargs={"enable_thinking": False},

    # Saving
    save_steps=50,
    save_total_limit=2,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_func,
    train_dataset=dataset,
    args=training_args,
    environment_factory=MaterialForgeTRLEnv,
)

print("Trainer initialized. Starting GRPO training...")
print(f"  Episodes: {NUM_EPISODES}")
print(f"  Generations per prompt: {training_args.num_generations}")
print(f"  Max completion length: {training_args.max_completion_length}")
print(f"  Temperature: {training_args.temperature}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Total steps: {NUM_EPISODES // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")

In [ ]:
train_result = trainer.train()
print(f"\nTraining complete!")
print(f"  Total steps: {trainer.state.global_step}")
print(f"  Final loss: {train_result.training_loss:.4f}")

## 9. Save Trained Model

In [ ]:
SAVE_DIR = "materialforge-grpo-trained"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}/")

## 10. Training Reward Curves

Extract and plot the reward history from training logs. These plots are a **required deliverable** — they must be saved as images and embedded in the README.

In [ ]:
import matplotlib.pyplot as plt
import json

os.makedirs("plots", exist_ok=True)

log_history = trainer.state.log_history

steps = []
rewards = []
losses = []

for entry in log_history:
    if "reward" in entry:
        steps.append(entry.get("step", len(steps)))
        rewards.append(entry["reward"])
    if "loss" in entry:
        if len(losses) < len(steps):
            losses.append(entry["loss"])

# --- Plot 1: Reward over training ---
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(steps[:len(rewards)], rewards, color="#4285f4", linewidth=2, label="GRPO Reward")
if len(rewards) > 5:
    window = max(3, len(rewards) // 10)
    smoothed = [sum(rewards[max(0,i-window):i+1])/len(rewards[max(0,i-window):i+1]) for i in range(len(rewards))]
    ax.plot(steps[:len(smoothed)], smoothed, color="#ea4335", linewidth=2, linestyle="--", label=f"Smoothed (window={window})")
ax.set_xlabel("Training Step", fontsize=12)
ax.set_ylabel("Reward", fontsize=12)
ax.set_title("MaterialForge GRPO Training — Reward Curve", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("plots/reward_curve.png", dpi=150)
plt.show()
print("Saved: plots/reward_curve.png")

# --- Plot 2: Loss over training ---
if losses:
    fig2, ax2 = plt.subplots(1, 1, figsize=(10, 5))
    ax2.plot(range(len(losses)), losses, color="#34a853", linewidth=2)
    ax2.set_xlabel("Training Step", fontsize=12)
    ax2.set_ylabel("Loss", fontsize=12)
    ax2.set_title("MaterialForge GRPO Training — Loss Curve", fontsize=14, fontweight="bold")
    ax2.grid(True, alpha=0.3)
    fig2.tight_layout()
    fig2.savefig("plots/loss_curve.png", dpi=150)
    plt.show()
    print("Saved: plots/loss_curve.png")

## 11. Baseline Comparison

Compare the trained agent against a random baseline to demonstrate improvement.

In [ ]:
import random as rng

def run_random_baseline(n_episodes=20, max_steps=30):
    """Run a random agent as baseline."""
    env = MaterialForgeEnvironment(rubric=HeuristicRewardRubric())
    episode_rewards = []
    episode_best = []

    for ep in range(n_episodes):
        obs = env.reset(seed=ep * 7, difficulty="easy")
        rewards = []
        for step in range(max_steps):
            action_type = rng.choice(["place", "place", "place", "replace", "remove"])
            row = rng.randint(0, 7)
            col = rng.randint(0, 7)
            atom = rng.choice(["A", "B", "C", "P"]) if action_type != "remove" else None
            action = MaterialForgeAction(action_type=action_type, row=row, col=col, atom=atom)
            obs = env.step(action)
            rewards.append(obs.reward if obs.reward else 0.0)
            if obs.done:
                break
        episode_rewards.append(sum(rewards) / len(rewards) if rewards else 0.0)
        episode_best.append(max(rewards) if rewards else 0.0)

    return episode_rewards, episode_best

print("Running random baseline (20 episodes)...")
random_avg_rewards, random_best_rewards = run_random_baseline(20)
print(f"Random Agent — Mean reward: {sum(random_avg_rewards)/len(random_avg_rewards):.4f}")
print(f"Random Agent — Mean best:   {sum(random_best_rewards)/len(random_best_rewards):.4f}")

In [ ]:
# Comparison bar chart
trained_mean_reward = sum(rewards) / len(rewards) if rewards else 0.0
random_mean_reward = sum(random_avg_rewards) / len(random_avg_rewards)

fig3, ax3 = plt.subplots(figsize=(8, 5))
agents = ["Random\nBaseline", "GRPO-Trained\nAgent"]
means = [random_mean_reward, trained_mean_reward]
colors = ["#ea4335", "#4285f4"]
bars = ax3.bar(agents, means, color=colors, width=0.5, edgecolor="white", linewidth=1.5)

for bar, val in zip(bars, means):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{val:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=13)

ax3.set_ylabel("Mean Reward", fontsize=12)
ax3.set_title("MaterialForge — Random Baseline vs GRPO-Trained Agent", fontsize=14, fontweight="bold")
ax3.set_ylim(0, max(means) * 1.3 + 0.05)
ax3.grid(axis="y", alpha=0.3)
fig3.tight_layout()
fig3.savefig("plots/baseline_comparison.png", dpi=150)
plt.show()
print("Saved: plots/baseline_comparison.png")

## 12. Trained Agent Inference Demo

Run the trained model on a fresh episode to see its crystal design strategy.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Loading trained model for inference...")
inf_model, inf_tokenizer = FastLanguageModel.from_pretrained(
    SAVE_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(inf_model)

demo_env = MaterialForgeTRLEnv()
obs_text = demo_env.reset()

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
    {"role": "assistant", "content": f"Let me analyze the initial state:\n{obs_text}\n\nI'll start placing atoms strategically."},
]

print("=" * 60)
print("TRAINED AGENT DEMO")
print("=" * 60)
print(f"\nInitial observation:\n{obs_text}")
print(f"\nAgent designing crystal...")
print(f"(Full multi-turn inference requires TRL's tool-call loop.")
print(f"See the training logs above for actual agent behavior.)")

## 13. Summary

| Metric | Random Baseline | GRPO-Trained Agent |
|--------|----------------|--------------------|
| Mean Reward | *see above* | *see above* |
| Phase Achieved | Amorphous | *see training logs* |

### Deliverables produced by this notebook:
- `plots/reward_curve.png` — Reward over training steps
- `plots/loss_curve.png` — Training loss curve
- `plots/baseline_comparison.png` — Random vs trained comparison
- `materialforge-grpo-trained/` — Saved model checkpoint

### Links
- **Live Environment**: [ArshPathan/material_forge_env](https://huggingface.co/spaces/ArshPathan/material_forge_env)
- **Repository**: [MaterialForge GitHub](https://github.com/Arsh-Pathan/MaterialForge)